In [1]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.ExtremeLearningMachine import ExtremeLearningMachine
from Pipeline.Methodology.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
model_configs = GlobalSetting.get_model_configs()

In [3]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [4]:
x_test_scaled   = gallstone_dataset.x_test_scaled
y_test          = gallstone_dataset.y_test
x_train_scaled  = gallstone_dataset.x_train_scaled
y_train         = gallstone_dataset.y_train

In [5]:
testing_results = []
for config in model_configs:
    for seed in GlobalSetting.elm_initial_state_range:
        elm = ExtremeLearningMachine(
            features_size   = features_size,
            hidden_size     = config["Hidden_Nodes"],
            activation_function     = config["Activation"],
            regularization_lambda   = config["Lambda_Value"]
        )
        elm.initialize_random_weights(random_seed = seed)

        elm.fit(x_train_scaled.values, y_train.values)
        y_pred = elm.predict(x_test_scaled.values)

        evaluation = EvaluationMatrix(y_test, y_pred)

        metrics = evaluation.get_all_metrics()
        test_result = {
            "Model_Type"   : config.get('Model_Types', 'Unknown'),
            "Hidden_Nodes" : config['Hidden_Nodes'],
            "Lambda_Value" : config['Lambda_Value'],
            "Activation"   : config['Activation'].__name__,
            "Seed"         : seed
        }
        test_result.update(metrics)
        testing_results.append(test_result)

In [6]:
# 1. 将字典列表转换为 DataFrame
df_results = pd.DataFrame(testing_results)

# 2. 定义哪些列代表一个唯一的 "Algorithm" 配置
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

# 3. 动态提取所有需要聚合的指标列（排除配置列和 Seed）
metric_cols = [col for col in df_results.columns if col not in group_cols + ["Seed"]]

# 4. 构造聚合字典：对每个指标求平均值(mean)和标准差(std)
agg_funcs = {metric: ['mean', 'std', 'max' , 'min'] for metric in metric_cols}

# 5. 执行 Group By
summary_df = df_results.groupby(group_cols).agg(agg_funcs).reset_index()

# 将多级表头 (例如 'Accuracy', 'mean') 拼接为单级表头 'Accuracy_mean'
summary_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df.columns.values
]

summary_df

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy_mean,Accuracy_std,Accuracy_max,Accuracy_min,Precision_mean,Precision_std,...,F2-Score_max,F2-Score_min,Bal Accuracy_mean,Bal Accuracy_std,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_std,MCC_max,MCC_min
0,Best_Hidden_Nodes,48,0.000000,sigmoid,0.760938,0.039390,0.828125,0.687500,0.770900,0.043020,...,0.848485,0.645161,0.760938,0.039390,0.828125,0.687500,0.523595,0.078820,0.656571,0.375000
1,Best_Lambda,48,0.031250,sigmoid,0.788021,0.034498,0.859375,0.734375,0.797561,0.036433,...,0.889571,0.705128,0.788021,0.034498,0.859375,0.734375,0.577242,0.069449,0.721930,0.468979
2,Best_Size_and_Lambda,48,0.031250,sigmoid,0.788021,0.034498,0.859375,0.734375,0.797561,0.036433,...,0.889571,0.705128,0.788021,0.034498,0.859375,0.734375,0.577242,0.069449,0.721930,0.468979
3,Grid_Combination,93,0.062500,sigmoid,0.801562,0.027265,0.859375,0.750000,0.809654,0.038535,...,0.864198,0.750000,0.801562,0.027265,0.859375,0.750000,0.604027,0.055569,0.719101,0.500000
4,Grid_Optimization,37,0.077887,sigmoid,0.781771,0.037938,0.843750,0.718750,0.792528,0.046431,...,0.884146,0.673077,0.781771,0.037938,0.843750,0.718750,0.565889,0.076638,0.692935,0.438357


In [7]:
GlobalSetting.save_dataframe_to_record(summary_df,"Model_Testing_Result.csv")

[I/O Trace] Record exported successfully: ../../Storage/Record\Model_Testing_Result.csv
